In [ ]:
import pandas as pd
import numpy as np

training_dataset = pd.read_csv('../data/raw/train.csv')
testing_dataset = pd.read_csv('../data/raw/test.csv')

train_target = training_dataset['health_condition']
train_ids = training_dataset['id']
test_ids = testing_dataset['id']

train_features = training_dataset.drop(columns=['id', 'health_condition'])
test_features = testing_dataset.drop(columns=['id'])

train_features['is_train'] = 1
test_features['is_train'] = 0

combined = pd.concat([train_features, test_features], axis=0, ignore_index=True)

print("Combined shape:", combined.shape)
print("\nColumns:", combined.columns.tolist())
print("\nDtypes:")
print(combined.dtypes)

In [ ]:
numeric_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 
                 'step_count', 'exercise_duration', 'water_intake']
categorical_cols = ['diet_type', 'stress_level', 'sleep_quality', 
                     'physical_activity_level', 'smoking_alcohol', 'gender']

for col in numeric_cols:
    median_val = combined[col].median()
    combined[col] = combined[col].fillna(median_val)
    print(f"{col}: filled with median = {median_val:.2f}")

for col in categorical_cols:
    combined[col] = combined[col].fillna('Missing')

print("\nMissing values after imputation:")
print(combined.isnull().sum().sum())

print("\nCategorical value counts (post-fill):")
for col in categorical_cols:
    print(f"\n{col}:")
    print(combined[col].value_counts())

In [ ]:
combined['stress_activity_combo'] = combined['stress_level'] + '_' + combined['physical_activity_level']

combined['sleep_duration_bin'] = pd.cut(combined['sleep_duration'], 
                                          bins=[0, 4, 6, 8, 12], 
                                          labels=['very_low', 'low', 'good', 'high'])

combined['bmi_category'] = pd.cut(combined['bmi'], 
                                    bins=[0, 18.5, 25, 30, 100], 
                                    labels=['underweight', 'normal', 'overweight', 'obese'])

combined['activity_score'] = combined['step_count'] / 1000 + combined['exercise_duration'] / 10

combined['calorie_per_step'] = combined['calorie_expenditure'] / (combined['step_count'] + 1)

print("New features created:")
print(combined[['stress_activity_combo', 'sleep_duration_bin', 'bmi_category', 
                 'activity_score', 'calorie_per_step']].head(10))

print("\nstress_activity_combo value counts:")
print(combined['stress_activity_combo'].value_counts())

In [ ]:
all_categorical_cols = categorical_cols + ['stress_activity_combo', 'sleep_duration_bin', 'bmi_category']

for col in all_categorical_cols:
    combined[col] = combined[col].astype('category')

print("Final dtypes:")
print(combined.dtypes)

print("\nFinal combined shape:", combined.shape)
print("\nSample rows:")
print(combined.head())

In [ ]:
import os

train_processed = combined[combined['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
test_processed = combined[combined['is_train'] == 0].drop(columns=['is_train']).reset_index(drop=True)

train_processed['id'] = train_ids.values
train_processed['health_condition'] = train_target.values

test_processed['id'] = test_ids.values

print("Train processed shape:", train_processed.shape)
print("Test processed shape:", test_processed.shape)

assert train_processed.shape[0] == training_dataset.shape[0]
assert test_processed.shape[0] == testing_dataset.shape[0]
print("\nRow counts verified ✓")

os.makedirs('../data/processed', exist_ok=True)
train_processed.to_csv('../data/processed/train_fe.csv', index=False)
test_processed.to_csv('../data/processed/test_fe.csv', index=False)

print("\nSaved to ../data/processed/")
print("Files:", os.listdir('../data/processed'))